# KcELECTRA fine-tuning: sentiment and rating–text mismatch (Korean e-commerce reviews)

Part of the *From Reviews to Reliable Actions* project. This notebook:
1. Fine-tunes **KcELECTRA** (a Korean ELECTRA trained on Naver comments) to predict whether a review text is positive or negative.
2. Uses **the same 80/20 split as `baseline.py`**, so its accuracy compares directly with the TF-IDF baseline (0.922).
3. Never trains on the 3,000 annotation-sample reviews, then scores them → `kcelectra_scores.csv` for `evaluate.py --pred`.

**How to run:** Runtime → Change runtime type → **T4 GPU**, then Runtime → Run all. Upload `sample_key_DO_NOT_SHARE.csv` when asked. About 20–30 minutes on a free T4.

In [ ]:
# Settings
MODEL_NAME = "beomi/KcELECTRA-base-v2022"
TRAIN_SIZE = 100_000   # reviews used for training (None = all ~160k). 100k is plenty and fits the free GPU time.
EPOCHS = 1
MAX_LEN = 64           # 75% of reviews are under 55 characters
BATCH = 64
LR = 5e-5
SEED = 42
DATA_URL = "https://raw.githubusercontent.com/bab2min/corpus/master/sentiment/naver_shopping.txt"

In [ ]:
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    os.system("pip -q install transformers")

In [ ]:
import json, random, time
import numpy as np, pandas as pd, torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from transformers import AutoTokenizer, AutoModelForSequenceClassification

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device, "| torch", torch.__version__)
if device == "cpu":
    print("WARNING: no GPU. In Colab: Runtime → Change runtime type → T4 GPU.")

## 1. Data — same cleaning and split as `baseline.py`

In [ ]:
if not os.path.exists("naver_shopping.txt"):
    os.system(f"wget -q -O naver_shopping.txt {DATA_URL}")
df = pd.read_csv("naver_shopping.txt", sep="\t", header=None, names=["rating", "text"], quoting=3)
df["text"] = df["text"].astype(str).str.strip()
df = df[df["text"].str.len() > 0].drop_duplicates(subset="text").reset_index(drop=True)
df["label"] = (df["rating"] >= 4).astype(int)
df["review_id"] = np.arange(len(df))
train, test = train_test_split(df, test_size=0.2, stratify=df["label"], random_state=SEED)
print(len(df), "reviews | train", len(train), "| test", len(test))

## 2. Upload the annotation key
`annotation/sample_key_DO_NOT_SHARE.csv` from the project folder. Its reviews are removed from training so the model never sees them.

In [ ]:
KEY = "sample_key_DO_NOT_SHARE.csv"
if IN_COLAB and not os.path.exists(KEY):
    from google.colab import files
    files.upload()
key = pd.read_csv(KEY)
train = train[~train["review_id"].isin(key["review_id"])]
if TRAIN_SIZE and len(train) > TRAIN_SIZE:
    train = train.sample(TRAIN_SIZE, random_state=SEED)
print("training on", len(train), "reviews (annotation sample excluded)")

## 3. Fine-tune

In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)

def batches(texts, labels=None, shuffle=False):
    idx = np.arange(len(texts))
    if shuffle:
        np.random.shuffle(idx)
    for i in range(0, len(idx), BATCH):
        j = idx[i:i + BATCH]
        enc = tok([texts[k] for k in j], padding=True, truncation=True, max_length=MAX_LEN, return_tensors="pt")
        enc = {k: v.to(device) for k, v in enc.items()}
        yield enc, (torch.tensor([labels[k] for k in j], device=device) if labels is not None else None)

tr_x, tr_y = train["text"].tolist(), train["label"].tolist()
steps = EPOCHS * ((len(tr_x) + BATCH - 1) // BATCH)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=steps, pct_start=0.06, anneal_strategy="linear")
use_amp = device == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

model.train(); step = 0; t0 = time.time()
for epoch in range(EPOCHS):
    for enc, y in batches(tr_x, tr_y, shuffle=True):
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
            loss = torch.nn.functional.cross_entropy(model(**enc).logits, y)
        opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        step += 1
        if step % 200 == 0 or step == steps:
            print(f"step {step}/{steps}  loss {loss.item():.4f}  {time.time() - t0:.0f}s")

## 4. Test accuracy — compare with the TF-IDF baseline (accuracy and macro F1 0.922)

In [ ]:
@torch.no_grad()
def p_positive(texts):
    model.eval(); out = []
    for enc, _ in batches(texts):
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
            out.append(torch.softmax(model(**enc).logits.float(), dim=-1)[:, 1].cpu().numpy())
    return np.concatenate(out)

p_test = p_positive(test["text"].tolist())
pred = (p_test >= 0.5).astype(int)
metrics = {"model": MODEL_NAME, "train_size": len(train), "epochs": EPOCHS,
           "accuracy": round(accuracy_score(test["label"], pred), 4),
           "macro_f1": round(f1_score(test["label"], pred, average="macro"), 4)}
print(json.dumps(metrics, indent=2))
print(classification_report(test["label"], pred, target_names=["negative", "positive"], digits=4))
json.dump(metrics, open("kcelectra_metrics.json", "w"), indent=2)

## 5. Score the 3,000 annotation reviews → `kcelectra_scores.csv`
Put this file in the project's `results/` folder, then run:
`python evaluate.py --labels annotation/claude_labels.csv --name claude --pred results/kcelectra_scores.csv`

In [ ]:
scores = key[["item", "review_id"]].copy()
scores["p_text_positive"] = p_positive(key["text"].astype(str).tolist()).round(4)
scores[["item", "p_text_positive"]].to_csv("kcelectra_scores.csv", index=False)
print(scores.head())
if IN_COLAB:
    from google.colab import files
    files.download("kcelectra_scores.csv"); files.download("kcelectra_metrics.json")